# Explosive NSE moves — master research notebook

Rare-event prediction: identify conditions **5–20 sessions** before unusually large upward price moves (MFE-based labels).

**Causal rule:** features at market close **T**; labels from **T+1…T+H** only.

**Order:** Phase 1 audit → panel → events → **pre-event discovery before ML** → features → ablation → walk-forward ML → strategy → scanner.

Artifacts: `results/explosive_moves/`. Override data path with `QUANT_DATA_DIR`.

**Limitations (v1):** no split-adjusted prices; sector/VIX/free-float only when datasets are added.


In [ ]:
import pickle
from quant.research.explosive_research import (
    trading_calendar_from_panel,
    calendar_shift,
    mark_ca_suspect,
    forward_excursion_calendar,
    ca_invalidates_forward_window,
    first_touch_triple_barrier,
    dedupe_event_episodes,
    daily_precision_lift,
    summarize_daily_metrics,
)

from pathlib import Path
import json, warnings, os, sys, gc, subprocess, importlib.util
warnings.filterwarnings("ignore")

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "explosive_moves"
for sub in ("panel", "events", "features", "models", "reports", "reports/figures", "scanner"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

PANEL_PATH = OUT_DIR / "panel" / "ohlcv_panel.parquet"
EVENTS_PATH = OUT_DIR / "events" / "explosive_events.parquet"
FEATURES_PATH = OUT_DIR / "features" / "feature_dataset.parquet"
AUDIT_PATH = OUT_DIR / "reports" / "data_audit.json"
PRE_EVENT_PATH = OUT_DIR / "reports" / "pre_event_summary.parquet"
ABLATION_PATH = OUT_DIR / "reports" / "feature_ablation.csv"
MODEL_EVAL_PATH = OUT_DIR / "reports" / "model_evaluation.csv"
SCANNER_PATH = OUT_DIR / "scanner" / "latest_ranked.parquet"
SCANNER_CSV = OUT_DIR / "scanner" / "latest_ranked.csv"

assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("NIFTY:", NIFTY50_PATH, "exists" if NIFTY50_PATH.exists() else "missing")


In [ ]:
required = ["pandas", "numpy", "pyarrow", "duckdb", "matplotlib", "scipy"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.options.mode.copy_on_write = True

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

HAS_LGBM = importlib.util.find_spec("lightgbm") is not None
if not HAS_LGBM:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        HAS_LGBM = True
    except Exception:
        HAS_LGBM = False
if HAS_LGBM:
    import lightgbm as lgb

print("duckdb:", duckdb.__version__, "| lightgbm:", HAS_LGBM)


## Phase 0 — Research configuration


In [ ]:
CFG = {
    "horizons": [5, 10, 20],
    "h_max": 20,
    "event_thresholds": {"E1": (0.10, 5), "E2": (0.20, 10), "E3": (0.20, 20)},
    "extra_thresholds": [0.40, 0.50, 1.00],
    "sustained_retention": {5: 0.05, 10: 0.10, 20: 0.10},
    "triple_barrier": {"up": 0.20, "down": -0.10},
    "min_price": 20.0,
    "min_history_bars": 60,
    "min_median_turnover_20d": 1_000_000.0,
    "eq_series_only": True,
    "top_n_events": 500,
    "pre_event_lags": [5, 10, 20],
    "event_episode_gap": 10,
    "match_k": 3,
    "random_seed": 42,
    "primary_horizon": 20,
    "primary_mfe_thresh": 0.20,
    "ca_gap_abs_exclude": 0.35,
    "wf_splits": {
        "train_end": 2018,
        "valid_end": 2020,
        "test1_end": 2022,
        "test2_end": 2024,
    },
    "costs": {
        "slippage_bps": 20.0,
        "brokerage_bps": 0.0,
        "stt_bps": 10.0,
        "exchange_bps": 3.25,
        "stamp_bps": 1.5,
        "sebi_bps": 0.0001,
        "gst_rate": 0.18,
    },
    "strategy": {
        "top_k": (10, 20),
        "prob_threshold": 0.25,
        "max_hold": 20,
        "stop_loss": 0.10,
        "max_participation_pct": 0.05,
        "initial_capital": 1_000_000.0,
    },
}

FEATURE_FLAGS = {
    "HAS_DELIVERY": False,
    "HAS_TURNOVER": False,
    "HAS_SERIES": False,
    "HAS_PREV_CLOSE": False,
}

print(json.dumps(CFG, indent=2))


## Phase 1 — Data audit


In [ ]:
def stock_parquet_glob():
    return sorted(
        p for p in DATA_DIR.rglob("nse_cm_*.parquet")
        if "results" not in p.parts and "indices" not in p.parts
    )

files = stock_parquet_glob()
print("Daily stock files:", len(files))
if not files:
    raise FileNotFoundError(f"No nse_cm_*.parquet under {DATA_DIR}")

schema_union = set()
for p in [files[0], files[len(files) // 2], files[-1]]:
    schema_union |= {c.lower() for c in pq.ParquetFile(p).schema.names}

con = duckdb.connect()
paths = [str(p).replace("'", "''") for p in files]
paths_sql = ", ".join(f"'{p}'" for p in paths)
audit_df = con.execute(
    f'''
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT symbol) AS n_symbols,
           MIN(CAST(date AS DATE)) AS dmin,
           MAX(CAST(date AS DATE)) AS dmax
    FROM read_parquet([{paths_sql}], union_by_name=true)
    '''
).df()
dup_df = con.execute(
    f'''
    SELECT date, symbol, COUNT(*) AS n
    FROM read_parquet([{paths_sql}], union_by_name=true)
    GROUP BY 1,2 HAVING n>1 LIMIT 10
    '''
).df()

FEATURE_FLAGS = {
    "HAS_TURNOVER": "turnover" in schema_union,
    "HAS_DELIVERY": ("deliv_per" in schema_union) or ("deliv_qty" in schema_union),
    "HAS_SERIES": "series" in schema_union,
    "HAS_PREV_CLOSE": "prev_close" in schema_union,
}
audit = {
    "n_files": len(files),
    "schema_union": sorted(schema_union),
    "feature_flags": FEATURE_FLAGS,
    "duckdb_summary": audit_df.to_dict(orient="records"),
    "duplicate_sample": dup_df.to_dict(orient="records"),
    "nifty_exists": NIFTY50_PATH.exists(),
}
AUDIT_PATH.write_text(json.dumps(audit, indent=2, default=str))
display(audit_df)
if not dup_df.empty:
    print("Duplicate (date,symbol) rows (sample):", len(dup_df))
print("Feature flags:", FEATURE_FLAGS)


## Phase 2 — Panel construction


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s


def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "tradingsymbol"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
        "turnover": ["turnover", "tottrdval", "traded_value"],
        "deliv_qty": ["deliv_qty", "deliverablequantity", "delivqty"],
        "deliv_per": ["deliv_per", "delivper", "delivery_percentage"],
        "series": ["series"],
        "prev_close": ["prev_close", "prevclose", "previous_close"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "Open": pd.to_numeric(first_series(df, out["open"]), errors="coerce") if "open" in out else np.nan,
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
        "Volume": pd.to_numeric(first_series(df, out["volume"]), errors="coerce") if "volume" in out else np.nan,
    })
    opt_map = {
        "turnover": "Turnover",
        "deliv_qty": "DelivQty",
        "deliv_per": "DelivPer",
        "prev_close": "PrevClose",
    }
    for src, dst in opt_map.items():
        if src in out:
            x[dst] = pd.to_numeric(first_series(df, out[src]), errors="coerce")
    if "series" in out:
        x["Series"] = first_series(df, out["series"]).astype(str).str.upper().str.strip()
    x = x.dropna(subset=["Date", "Close"]).loc[x["Close"] > 0]
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    for c in x.select_dtypes(include=["float64"]).columns:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x


PANEL_EXTRA = ["Turnover", "DelivQty", "DelivPer", "Series", "PrevClose"]
BASE_COLS = ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]


PANEL_META_PATH = OUT_DIR / "panel" / "panel_build_meta.json"


def panel_source_fingerprint(file_paths):
    if not file_paths:
        return ""
    mtimes = sorted((p.stat().st_mtime, str(p)) for p in file_paths)
    return json.dumps({"n_files": len(file_paths), "first": mtimes[0][1], "last": mtimes[-1][1], "sum_mtime": sum(m for m, _ in mtimes)})


def rebuild_panel(force=False):
    files = stock_parquet_glob()
    fp = panel_source_fingerprint(files)
    meta = {}
    if PANEL_META_PATH.exists():
        meta = json.loads(PANEL_META_PATH.read_text())
    if PANEL_PATH.exists() and not force and meta.get("fingerprint") == fp:
        print("Using existing panel (fingerprint match):", PANEL_PATH)
        return
    writer = None
    n_ok = n_err = 0
    for i, f in enumerate(files, 1):
        try:
            x = normalize_ohlc(pd.read_parquet(f), f.name)
            keep = [c for c in BASE_COLS + PANEL_EXTRA if c in x.columns]
            table = pa.Table.from_pandas(x[keep], preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, table.schema, compression="zstd")
            writer.write_table(table)
            n_ok += 1
            if i % 400 == 0:
                gc.collect()
                print(f"stitched {i}/{len(files)}")
        except Exception as exc:
            n_err += 1
            if n_err <= 5:
                print("skip", f.name, exc)
    if writer:
        writer.close()
    PANEL_META_PATH.write_text(json.dumps({"fingerprint": fp, "n_files": len(files), "n_ok": n_ok, "n_err": n_err}, indent=2))
    print("Panel OK:", n_ok, "errors:", n_err)


rebuild_panel(force=False)
panel = pd.read_parquet(PANEL_PATH)
panel["Date"] = pd.to_datetime(panel["Date"])
panel["Symbol"] = panel["Symbol"].astype("string")
panel = panel.sort_values(["Symbol", "Date"])
n_dup = int(panel.duplicated(["Date", "Symbol"]).sum())
if n_dup:
    print("Dropping duplicate Date-Symbol rows:", n_dup)
    panel = panel.drop_duplicates(["Date", "Symbol"], keep="last")
panel = panel.reset_index(drop=True)

if "Turnover" not in panel.columns:
    panel["Turnover"] = (panel["Close"] * panel["Volume"]).astype("float32")

if CFG["eq_series_only"] and "Series" in panel.columns:
    bad = panel["Series"].notna() & ~panel["Series"].str.upper().eq("EQ")
    panel = panel.loc[~bad]

nifty = None
if NIFTY50_PATH.exists():
    nifty = normalize_ohlc(pd.read_parquet(NIFTY50_PATH), "NIFTY50")
    nifty = nifty.sort_values("Date")
    nifty["nifty_ret_1"] = nifty["Close"].pct_change()

print(panel.shape, panel["Date"].min(), "→", panel["Date"].max(), "symbols:", panel["Symbol"].nunique())


## Phase 3 — Event labeling (MFE / MAE / forward close return)


In [ ]:
def mfe_threshold_for_horizon(H, cfg):
    thr_map = {h: t for t, h in cfg["event_thresholds"].values()}
    if H == cfg["primary_horizon"]:
        return cfg["primary_mfe_thresh"]
    return thr_map.get(H, cfg["primary_mfe_thresh"])


def add_labels(df, cfg):
    df = df.copy()
    ca = df["ca_suspect"].fillna(False)
    for H in cfg["horizons"]:
        thr = mfe_threshold_for_horizon(H, cfg)
        mfe, ret = df[f"mfe_{H}"], df[f"ret_{H}"]
        ok = mfe.notna() & df[f"horizon_complete_{H}"] & ~df[f"ca_fwd_invalid_{H}"]
        df[f"y_explosive_{H}"] = ((mfe >= thr) & ok).astype("int8")
        sustain = cfg["sustained_retention"].get(H, 0.10)
        df[f"y_sustained_{H}"] = ((mfe >= thr) & (ret >= sustain) & ok).astype("int8")
        # y_tb already computed per symbol
    return df


TRADING_CAL = trading_calendar_from_panel(panel)

labeled_parts = []
for sym, g in panel.groupby("Symbol", observed=True):
    if len(g) < CFG["min_history_bars"] + CFG["h_max"] + 1:
        continue
    g = forward_excursion_calendar(g, TRADING_CAL, CFG["horizons"])
    ca = mark_ca_suspect(g, CFG["ca_gap_abs_exclude"])
    g["ca_suspect"] = ca.values
    for H in CFG["horizons"]:
        g[f"ca_fwd_invalid_{H}"] = ca_invalidates_forward_window(g, TRADING_CAL, H, ca).values
        ok = g[f"horizon_complete_{H}"] & ~g[f"ca_fwd_invalid_{H}"]
        g[f"y_tb_{H}"] = first_touch_triple_barrier(
            g, TRADING_CAL, H, CFG["triple_barrier"]["up"], CFG["triple_barrier"]["down"]
        ).where(ok, np.nan)
    labeled_parts.append(g)

labeled = pd.concat(labeled_parts, ignore_index=True)
labeled = add_labels(labeled, CFG)
H = CFG["primary_horizon"]
mfe = labeled[f"mfe_{H}"]
ok = mfe.notna() & labeled[f"horizon_complete_{H}"] & ~labeled[f"ca_fwd_invalid_{H}"]
for xt in CFG["extra_thresholds"]:
    labeled[f"y_mfe{int(xt*100)}_{H}"] = ((mfe >= xt) & ok).astype("int8")
labeled["ca_excluded"] = labeled["ca_suspect"].astype("int8")

print("Primary threshold:", mfe_threshold_for_horizon(CFG["primary_horizon"], CFG))
print("Incomplete horizons:", int((~labeled[f"horizon_complete_{CFG['h_max']}"]).sum()))
print("CA-forward invalid:", int(labeled[f"ca_fwd_invalid_{CFG['h_max']}"].sum()))

events = labeled.dropna(subset=[f"mfe_{CFG['primary_horizon']}"])
events.to_parquet(EVENTS_PATH, compression="zstd", index=False)
print("Wrote", EVENTS_PATH, len(events))



## Phase 5 — Feature dataset (stock-day rows)


In [ ]:
def compute_panel_features(panel_df, nifty_df=None):
    """Rolling features on the full tradable panel (labels merged later)."""
    x = panel_df.sort_values(["Symbol", "Date"]).copy()
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce")
    g = x.groupby("Symbol", observed=True)

    x["vol_rvol_20"] = x["Volume"] / g["Volume"].transform(lambda s: s.rolling(20, min_periods=10).mean())
    x["vol_va"] = g["Volume"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Volume"].transform(
        lambda s: s.rolling(20, min_periods=10).mean()
    )
    x["vol_turn_accel"] = g["Turnover"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["Turnover"].transform(
        lambda s: s.rolling(20, min_periods=10).mean()
    )

    prev_c = g["Close"].shift(1)
    tr = np.maximum(x["High"] - x["Low"], np.maximum((x["High"] - prev_c).abs(), (x["Low"] - prev_c).abs()))
    x["vola_atr14_pct"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(14, min_periods=5).mean()) / x["Close"] * 100
    x["vola_atr5"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(5, min_periods=3).mean())
    x["vola_atr50"] = tr.groupby(x["Symbol"]).transform(lambda s: s.rolling(50, min_periods=20).mean())
    x["vola_vr"] = x["vola_atr5"] / x["vola_atr50"]
    roll_std = g["Close"].transform(lambda s: s.rolling(20, min_periods=10).std())
    x["vola_bbw"] = (4 * roll_std) / g["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())

    x["px_ret_5"] = g["Close"].pct_change(5)
    x["px_ret_10"] = g["Close"].pct_change(10)
    x["px_ret_20"] = g["Close"].pct_change(20)
    x["px_dist_20h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(20, min_periods=10).max()) - 1
    x["px_dist_50h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(50, min_periods=20).max()) - 1
    x["px_dist_252h"] = x["Close"] / g["High"].transform(lambda s: s.rolling(252, min_periods=60).max()) - 1
    x["px_clv"] = (x["Close"] - x["Low"]) / (x["High"] - x["Low"]).replace(0, np.nan)
    up = (x["Close"] > g["Close"].shift(1)).astype(int)
    x["px_up_streak"] = up.groupby(x["Symbol"]).transform(lambda s: s.groupby((s != s.shift()).cumsum()).cumsum())

    if "DelivPer" in x.columns:
        x["deliv_ma5"] = g["DelivPer"].transform(lambda s: s.rolling(5, min_periods=3).mean())
        x["deliv_ma20"] = g["DelivPer"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        x["deliv_accel"] = x["deliv_ma5"] / x["deliv_ma20"]
    if "DelivQty" in x.columns:
        x["deliv_qty_accel"] = g["DelivQty"].transform(lambda s: s.rolling(5, min_periods=3).mean()) / g["DelivQty"].transform(
            lambda s: s.rolling(20, min_periods=10).mean()
        )

    x["liq_turn_med20"] = g["Turnover"].transform(lambda s: s.rolling(20, min_periods=10).median())
    # stale prints / zero-return days (not a pure illiquidity proxy)
    x["liq_zero_ret_20"] = g["Close"].pct_change().transform(
        lambda s: s.rolling(20, min_periods=10).apply(lambda a: np.mean(np.abs(a) < 1e-8), raw=True)
    )
    if "PrevClose" in x.columns:
        x["liq_gap"] = x["Close"] / x["PrevClose"] - 1

    if nifty_df is not None:
        nd = nifty_df[["Date", "Close"]].rename(columns={"Close": "NiftyClose"})
        nd["Date"] = pd.to_datetime(nd["Date"], errors="coerce")
        x = x.merge(nd, on="Date", how="left")
        nidx = nd.set_index("Date")["NiftyClose"]
        x["mkt_nifty_ret_5"] = nidx.pct_change(5).reindex(x["Date"]).to_numpy()
        x["mkt_nifty_ret_20"] = nidx.pct_change(20).reindex(x["Date"]).to_numpy()
        x["rs_20"] = x["px_ret_20"] - x["mkt_nifty_ret_20"]
        sma20 = g["Close"].transform(lambda s: s.rolling(20, min_periods=10).mean())
        x_day = x.assign(_above=(x["Close"] > sma20).astype(int))
        breadth = x_day.groupby("Date")["_above"].mean().rename("mkt_breadth_20sma")
        x = x.merge(breadth, on="Date", how="left")
    return x


def attach_event_labels(feat_df, events_df):
    label_cols = [c for c in events_df.columns if c.startswith(("mfe_", "mae_", "ret_", "y_", "horizon_", "ca_"))]
    out = feat_df.merge(events_df[["Date", "Symbol"] + label_cols], on=["Date", "Symbol"], how="left")
    out = out.loc[out["Close"] >= CFG["min_price"]]
    out = out.loc[out["liq_turn_med20"] >= CFG["min_median_turnover_20d"]]
    return out


panel_features = compute_panel_features(panel, nifty)
feature_cols = [
    c for c in panel_features.columns if c.startswith(("vol_", "vola_", "px_", "deliv_", "liq_", "rs_", "mkt_"))
]
features = attach_event_labels(panel_features, events)
label_cols = [c for c in features.columns if c.startswith("y_")]
print("panel_features:", panel_features.shape, "| labeled rows:", len(features), "features:", len(feature_cols))
features.to_parquet(FEATURES_PATH, compression="zstd", index=False)
PANEL_FEATURES_PATH = OUT_DIR / "features" / "panel_features.parquet"
panel_features.to_parquet(PANEL_FEATURES_PATH, compression="zstd", index=False)
print("Wrote", FEATURES_PATH, "and", PANEL_FEATURES_PATH)


## Phase 4 — Pre-event discovery (after Phase 5)


In [ ]:
rng = np.random.default_rng(CFG["random_seed"])
H = CFG["primary_horizon"]
mfe_col, y_col = f"mfe_{H}", f"y_explosive_{H}"

event_pool = events.loc[
    events[mfe_col].notna()
    & events[f"horizon_complete_{H}"]
    & ~events[f"ca_fwd_invalid_{H}"]
    & (events[mfe_col] >= CFG["primary_mfe_thresh"])
].copy()

candidates = dedupe_event_episodes(
    event_pool,
    mfe_col=mfe_col,
    calendar=TRADING_CAL,
    gap_sessions=CFG["event_episode_gap"],
).sort_values(mfe_col, ascending=False).head(CFG["top_n_events"])

if candidates.empty:
    raise ValueError("No explosive events after dedup; check panel / thresholds.")


def liquidity_bucket(turnover):
    if not np.isfinite(turnover) or turnover <= 0:
        return "unk"
    return ["micro", "small", "mid", "large"][min(3, int(np.log10(turnover + 1) // 2))]


candidates["liq_bucket"] = candidates["Turnover"].map(liquidity_bucket)
pool = events.loc[events["Close"] >= CFG["min_price"]]
used_controls: set[tuple] = set()
LAGS = CFG["pre_event_lags"]
FEATURES_PRE = [c for c in feature_cols if c in panel_features.columns]

feat_index = panel_features.set_index(["Date", "Symbol"])


def match_controls(evt_row, k=3):
    same_day = pool.loc[pool["Date"] == evt_row["Date"]]
    same_day = same_day.loc[same_day[y_col] == 0]
    if same_day.empty:
        return pd.DataFrame()
    lo, hi = evt_row["Close"] * 0.5, evt_row["Close"] * 2.0
    same_day = same_day.loc[(same_day["Close"] >= lo) & (same_day["Close"] <= hi)]
    b = evt_row["liq_bucket"]
    same_day = same_day.loc[same_day["Turnover"].map(liquidity_bucket) == b]
    if same_day.empty:
        same_day = pool.loc[(pool["Date"] == evt_row["Date"]) & (pool[y_col] == 0)]
    same_day = same_day.loc[~same_day.apply(lambda r: (r["Date"], r["Symbol"]) in used_controls, axis=1)]
    if len(same_day) <= k:
        pick = same_day
    else:
        pick = same_day.sample(n=k, random_state=rng.integers(0, 2**31 - 1))
    for _, r in pick.iterrows():
        used_controls.add((r["Date"], r["Symbol"]))
    return pick


cohort_rows = []
for _, ev in candidates.iterrows():
    ctl = match_controls(ev, CFG["match_k"])
    if ctl.empty:
        continue
    for lag in LAGS:
        ev_asof = calendar_shift(TRADING_CAL, pd.Series([ev["Date"]]), -lag).iloc[0]
        if pd.isna(ev_asof):
            continue
        key = (pd.Timestamp(ev_asof), ev["Symbol"])
        if key not in feat_index.index:
            continue
        ev_rec = {"cohort": "event", "lag": lag, "event_date": ev["Date"], "asof_date": ev_asof, "Symbol": ev["Symbol"]}
        for c in FEATURES_PRE:
            ev_rec[c] = feat_index.loc[key, c]
        cohort_rows.append(ev_rec)
        for _, cr in ctl.iterrows():
            ctl_asof = calendar_shift(TRADING_CAL, pd.Series([cr["Date"]]), -lag).iloc[0]
            if pd.isna(ctl_asof):
                continue
            ckey = (pd.Timestamp(ctl_asof), cr["Symbol"])
            if ckey not in feat_index.index:
                continue
            ctl_rec = {"cohort": "control", "lag": lag, "event_date": ev["Date"], "asof_date": ctl_asof, "Symbol": cr["Symbol"]}
            for c in FEATURES_PRE:
                ctl_rec[c] = feat_index.loc[ckey, c]
            cohort_rows.append(ctl_rec)

cohort = pd.DataFrame(cohort_rows)
if cohort.empty:
    raise ValueError("Pre-event cohort empty; check panel_features.")

n_tests = len(FEATURES_PRE) * len(LAGS)
tests = []
for lag in LAGS:
    for fc in FEATURES_PRE:
        ev_s = pd.to_numeric(cohort.loc[(cohort["cohort"] == "event") & (cohort["lag"] == lag), fc], errors="coerce")
        ctl_s = pd.to_numeric(cohort.loc[(cohort["cohort"] == "control") & (cohort["lag"] == lag), fc], errors="coerce")
        a = ev_s.to_numpy(dtype=np.float64)
        b = ctl_s.to_numpy(dtype=np.float64)
        a, b = a[np.isfinite(a)], b[np.isfinite(b)]
        if len(a) < 30 or len(b) < 30:
            continue
        stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        tests.append({
            "lag": lag,
            "feature": fc,
            "n_events": len(a),
            "n_controls": len(b),
            "event_median": float(np.median(a)),
            "control_median": float(np.median(b)),
            "u_stat": float(stat),
            "p_value": float(p),
            "p_bonferroni": min(1.0, float(p) * n_tests),
        })

pre_tests = pd.DataFrame(tests).sort_values("p_bonferroni")
pre_tests.to_parquet(PRE_EVENT_PATH, compression="zstd", index=False)
cohort.to_parquet(OUT_DIR / "reports" / "pre_event_cohort.parquet", compression="zstd", index=False)
print("Deduped events:", len(candidates), "| cohort rows:", len(cohort), "| tests:", len(pre_tests))
display(pre_tests.head(20))


## Phase 6 — Controlled experiments (feature-family ablation)


In [ ]:
FAMILIES = {
    "E1_volume": [c for c in feature_cols if c.startswith("vol_")],
    "E2_volatility": [c for c in feature_cols if c.startswith("vola_")],
    "E3_delivery": [c for c in feature_cols if c.startswith("deliv_")],
    "E4_price_structure": [c for c in feature_cols if c.startswith("px_")],
    "E5_relative_strength": [c for c in feature_cols if c.startswith("rs_")],
    "E6_liquidity": [c for c in feature_cols if c.startswith("liq_")],
    "E7_market": [c for c in feature_cols if c.startswith("mkt_")],
}

TARGET = f"y_explosive_{CFG['primary_horizon']}"


def split_masks(df):
    y = df["Date"].dt.year
    s = CFG["wf_splits"]
    train = y <= s["train_end"]
    valid = (y > s["train_end"]) & (y <= s["valid_end"])
    test1 = (y > s["valid_end"]) & (y <= s["test1_end"])
    test2 = (y > s["test1_end"]) & (y <= s["test2_end"])
    holdout = y > s["test2_end"]
    return train, valid, test1, test2, holdout


def purge_overlaps_calendar(train_mask, test_start, horizon, calendar):
    """Drop training rows whose forward label window could overlap the test start (session-based)."""
    test_start = pd.Timestamp(test_start)
    pos = calendar.get_indexer([test_start])[0]
    if pos < 0:
        return train_mask
    cutoff = calendar[max(0, pos - horizon)]
    dates = features.loc[train_mask, "Date"]
    return train_mask & (dates <= cutoff)


def precision_at_k(y_true, y_score, k=20):
    if len(y_true) == 0:
        return np.nan
    k = min(k, len(y_true))
    idx = np.argsort(-y_score)[:k]
    return float(y_true.iloc[idx].mean())


def lift_at_k(y_true, y_score, k=20):
    base = y_true.mean()
    if base <= 0:
        return np.nan
    return precision_at_k(y_true, y_score, k) / base


def fit_logit_scorer(train_df, cols, target):
    cols = [c for c in cols if c in train_df.columns]
    if not cols:
        return None, cols
    tr = train_df.dropna(subset=cols + [target])
    if tr[target].sum() < 5:
        return None, cols
    X = tr[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = tr[target]
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    w = len(y) / max(y.sum(), 1)
    clf = LogisticRegression(max_iter=500, class_weight={0: 1, 1: w})
    clf.fit(Xs, y)
    return (scaler, clf), cols


def score_df(df, model, cols, target):
    if model is None:
        return pd.Series(np.zeros(len(df)), index=df.index)
    scaler, clf = model
    X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return pd.Series(clf.predict_proba(scaler.transform(X))[:, 1], index=df.index)


def eval_daily_scanner(frame, scores, target, k=20):
    tmp = frame[["Date", target]].copy()
    tmp["score"] = scores
    daily = daily_precision_lift(tmp, date_col="Date", score_col="score", y_col=target, k=k)
    return daily, summarize_daily_metrics(daily, k)


train_m, valid_m, test1_m, test2_m, holdout_m = split_masks(features)

train_ab = features.loc[train_m]
valid_ab = features.loc[valid_m]
all_cols = feature_cols

full_model, full_cols = fit_logit_scorer(train_ab, all_cols, TARGET)
scores = score_df(valid_ab, full_model, full_cols, TARGET)
y = valid_ab[TARGET]
_, full_daily = eval_daily_scanner(valid_ab, scores, TARGET, k=20)

rows = []
for fam, cols in FAMILIES.items():
    ablate_cols = [c for c in all_cols if c not in cols]
    m, used = fit_logit_scorer(train_ab, ablate_cols, TARGET)
    sc = score_df(valid_ab, m, used, TARGET)
    _, daily = eval_daily_scanner(valid_ab, sc, TARGET, k=20)
    rows.append({
        "experiment": fam,
        "mean_precision_at_20": daily["mean_precision_at_20"],
        "mean_lift_at_20": daily["mean_lift_at_20"],
        "pct_days_with_hit": daily["pct_days_with_hit"],
        "delta_lift_vs_full": daily["mean_lift_at_20"] - full_daily["mean_lift_at_20"],
        "n_features_removed": len(cols),
    })
rows.append({
    "experiment": "full",
    "mean_precision_at_20": full_daily["mean_precision_at_20"],
    "mean_lift_at_20": full_daily["mean_lift_at_20"],
    "pct_days_with_hit": full_daily["pct_days_with_hit"],
    "delta_lift_vs_full": 0.0,
    "n_features_removed": 0,
})
ablation = pd.DataFrame(rows)
ablation.to_csv(ABLATION_PATH, index=False)
display(ablation)


## Phase 7 — ML + purged walk-forward


In [ ]:
from sklearn.isotonic import IsotonicRegression

MIN_TRAIN_POS = 20
Xcols = [c for c in feature_cols if c in features.columns]
TARGET = f"y_explosive_{CFG['primary_horizon']}"
MODEL_BUNDLE_PATH = OUT_DIR / "models" / "primary_model_bundle.pkl"
calibrator = None


def prep_xy(frame):
    f = frame.dropna(subset=Xcols + [TARGET])
    X = f[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = f[TARGET]
    return X, y, f


def eval_probs_daily(frame, scores, tag, k=20):
    y = frame[TARGET]
    _, summ = eval_daily_scanner(frame, scores, TARGET, k=k)
    out = {"split": tag, "pr_auc": float(average_precision_score(y, scores)) if y.sum() > 0 else np.nan}
    out.update(summ)
    out["brier"] = float(brier_score_loss(y, scores))
    out["base_rate"] = float(y.mean())
    out["n"] = int(len(y))
    return out


results = []
models = {}

# Purged train for the primary model; calibrate only on chronological validation.
tr_mask = purge_overlaps_calendar(train_m, features.loc[valid_m, "Date"].min(), CFG["h_max"], TRADING_CAL)
Xtr, ytr, _ = prep_xy(features.loc[tr_mask])
Xva, yva, fva = prep_xy(features.loc[valid_m])

if ytr.sum() >= MIN_TRAIN_POS and len(Xva) > 50 and HAS_LGBM:
    scale = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
    primary_lgbm = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight={0: 1.0, 1: scale},
        random_state=CFG["random_seed"],
    )
    primary_lgbm.fit(Xtr, ytr)
    p_va_raw = primary_lgbm.predict_proba(Xva)[:, 1]
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(p_va_raw, yva)
    models["lgbm"] = primary_lgbm

for name, mask in [("valid", valid_m), ("test1", test1_m), ("test2", test2_m), ("holdout", holdout_m)]:
    tr_mask = train_m
    if name != "valid":
        tr_mask = train_m | valid_m
    test_start = features.loc[mask, "Date"].min()
    if pd.notna(test_start):
        tr_mask = purge_overlaps_calendar(tr_mask, test_start, CFG["h_max"], TRADING_CAL)
    Xtr, ytr, _ = prep_xy(features.loc[tr_mask])
    Xte, yte, fte = prep_xy(features.loc[mask])
    if len(Xtr) < 100 or len(Xte) < 50:
        continue
    if ytr.sum() < MIN_TRAIN_POS:
        print(f"skip {name}: only {int(ytr.sum())} positives in training")
        continue

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    logit = LogisticRegression(max_iter=800, class_weight="balanced")
    logit.fit(Xtr_s, ytr)
    p_logit = logit.predict_proba(Xte_s)[:, 1]
    results.append(eval_probs_daily(fte, pd.Series(p_logit, index=fte.index), f"logit_{name}"))

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=CFG["random_seed"],
        n_jobs=-1,
    )
    rf.fit(Xtr, ytr)
    p_rf = rf.predict_proba(Xte)[:, 1]
    results.append(eval_probs_daily(fte, pd.Series(p_rf, index=fte.index), f"rf_{name}"))

    if models.get("lgbm") is not None and calibrator is not None:
        p_raw = models["lgbm"].predict_proba(Xte)[:, 1]
        p_lgb = calibrator.predict(p_raw)
        results.append(eval_probs_daily(fte, pd.Series(p_lgb, index=fte.index), f"lgbm_{name}"))

if models.get("lgbm") is not None and calibrator is not None:
    bundle = {
        "model": models["lgbm"],
        "calibrator": calibrator,
        "feature_cols": Xcols,
        "target": TARGET,
        "cfg_snippet": {
            "primary_horizon": CFG["primary_horizon"],
            "prob_threshold": CFG["strategy"]["prob_threshold"],
        },
    }
    MODEL_BUNDLE_PATH.write_bytes(pickle.dumps(bundle))
    models["lgbm"].booster_.save_model(str(OUT_DIR / "models" / "lgbm_primary.txt"))
    print("Saved bundle:", MODEL_BUNDLE_PATH)

model_eval = pd.DataFrame(results)
model_eval.to_csv(MODEL_EVAL_PATH, index=False)
display(model_eval)


## Phase 8 — Executable strategy simulation (research)


In [ ]:
def transaction_cost(notional, side, cfg):
    c = cfg["costs"]
    brokerage = notional * c["brokerage_bps"] / 10000
    stt = notional * c["stt_bps"] / 10000 if side == "SELL" else 0
    exchange = notional * c["exchange_bps"] / 10000
    stamp = notional * c["stamp_bps"] / 10000 if side == "BUY" else 0
    sebi = notional * c["sebi_bps"] / 10000
    gst = (brokerage + exchange + sebi) * c["gst_rate"]
    return brokerage + stt + exchange + stamp + sebi + gst


def execution_price(px, side, cfg):
    slip = cfg["costs"]["slippage_bps"] / 10000
    return px * (1 + slip) if side == "BUY" else px * (1 - slip)


def sessions_held(entry_date, current_date, calendar):
    i0 = calendar.get_indexer([pd.Timestamp(entry_date)])[0]
    i1 = calendar.get_indexer([pd.Timestamp(current_date)])[0]
    if i0 < 0 or i1 < 0:
        return 0
    return max(0, i1 - i0)


def simulate_topk_calendar(features_df, panel_df, scores, top_k, cfg, calendar):
    st = cfg["strategy"]
    f = features_df.copy()
    f["score"] = scores
    f["Date"] = pd.to_datetime(f["Date"])
    dmin, dmax = f["Date"].min(), f["Date"].max()
    dates = [d for d in calendar if dmin <= d <= dmax]
    if len(dates) < 2:
        return pd.DataFrame(), {}

    cash = st["initial_capital"]
    positions = {}
    equity = []
    trades = 0

    for i in range(len(dates) - 1):
        d, nxt = dates[i], dates[i + 1]
        panel_nxt = panel_df.loc[panel_df["Date"] == nxt].set_index("Symbol")

        for sym in list(positions):
            pos = positions[sym]
            if sym not in panel_nxt.index:
                continue
            row = panel_nxt.loc[sym]
            opn, lo = float(row["Open"]), float(row["Low"])
            held = sessions_held(pos["entry_date"], nxt, calendar)
            stop = pos["entry_px"] * (1 - st["stop_loss"])
            sell = False
            if np.isfinite(lo) and lo <= stop:
                sell = True
                raw = min(opn, stop)
            elif held >= st["max_hold"]:
                sell = True
                raw = opn
            if sell and np.isfinite(raw) and raw > 0:
                px = execution_price(raw, "SELL", cfg)
                cash += pos["shares"] * px - transaction_cost(pos["shares"] * px, "SELL", cfg)
                del positions[sym]
                trades += 1

        day = f.loc[f["Date"] == d].nlargest(top_k, "score")
        slots = top_k - len(positions)
        for _, r in day.iterrows():
            if slots <= 0:
                break
            sym = r["Symbol"]
            if sym in positions:
                continue
            if sym not in panel_nxt.index:
                continue
            nxt_row = panel_nxt.loc[sym]
            opn = float(nxt_row["Open"])
            prev_c = float(r["Close"])
            if not np.isfinite(opn) or opn <= 0:
                continue
            if prev_c > 0 and opn >= prev_c * 1.195:
                continue
            vol = float(nxt_row["Volume"]) if np.isfinite(nxt_row["Volume"]) else 0
            max_notional = vol * opn * st["max_participation_pct"]
            alloc = cash / max(slots, 1)
            notional = min(alloc, max_notional) if max_notional > 0 else alloc
            px = execution_price(opn, "BUY", cfg)
            shares = notional / px
            cost = transaction_cost(notional, "BUY", cfg)
            if cash < notional + cost:
                continue
            cash -= notional + cost
            positions[sym] = {"entry_px": px, "shares": shares, "entry_date": nxt, "prev_close": prev_c}
            slots -= 1
            trades += 1

        mtm = cash
        for sym, pos in positions.items():
            if sym in panel_nxt.index:
                mtm += pos["shares"] * float(panel_nxt.loc[sym, "Close"])
        equity.append({"Date": nxt, "equity": mtm, "n_pos": len(positions)})

    eq = pd.DataFrame(equity)
    if eq.empty:
        return eq, {}
    peak = eq["equity"].cummax()
    dd = (eq["equity"] / peak - 1).min()
    ret = eq["equity"].iloc[-1] / st["initial_capital"] - 1
    rets = eq["equity"].pct_change().dropna()
    sharpe = float(rets.mean() / rets.std() * np.sqrt(252)) if len(rets) > 1 and rets.std() > 0 else np.nan
    stats_out = {
        "total_return": float(ret),
        "max_drawdown": float(dd),
        "sharpe_daily_ann": sharpe,
        "n_trades": trades,
        "n_sessions": len(eq),
    }
    return eq, stats_out


if "lgbm" in models and calibrator is not None:
    test = features.loc[test2_m | holdout_m].copy()
    Xte = test[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    test_scores = calibrator.predict(models["lgbm"].predict_proba(Xte)[:, 1])
    for k in CFG["strategy"]["top_k"]:
        eq, stout = simulate_topk_calendar(test, panel, test_scores, k, CFG, TRADING_CAL)
        if not eq.empty:
            print(f"Top-{k} sim:", {kk: (f"{vv:.2%}" if kk == "total_return" else vv) for kk, vv in stout.items()})
            eq.to_parquet(OUT_DIR / "reports" / f"equity_top{k}.parquet", index=False)
            Path(OUT_DIR / "reports" / f"equity_top{k}_stats.json").write_text(json.dumps(stout, indent=2))
else:
    print("Calibrated LightGBM unavailable; skip strategy simulation.")


## Phase 9 — Daily scanner export


In [ ]:
latest_date = panel_features["Date"].max()
scan = panel_features.loc[panel_features["Date"] == latest_date].copy()

prob_thresh = CFG["strategy"]["prob_threshold"]

if "lgbm" in models and calibrator is not None:
    X = scan[Xcols].replace([np.inf, -np.inf], np.nan).fillna(0)
    scan["prob_explosive"] = calibrator.predict(models["lgbm"].predict_proba(X)[:, 1])
elif full_model is not None:
    scan["prob_explosive"] = score_df(scan, full_model, full_cols, TARGET)
else:
    scan["prob_explosive"] = np.nan

if np.isfinite(prob_thresh):
    scan = scan.loc[scan["prob_explosive"] >= prob_thresh]

rank_cols = ["Date", "Symbol", "Close", "prob_explosive", "vol_rvol_20", "vola_vr", "px_dist_252h", "rs_20"]
rank_cols = [c for c in rank_cols if c in scan.columns]
scan = scan.sort_values("prob_explosive", ascending=False)
scan[rank_cols].head(50).to_csv(SCANNER_CSV, index=False)
scan.to_parquet(SCANNER_PATH, compression="zstd", index=False)
print("Scanner date", latest_date, "rows:", len(scan), "→", SCANNER_PATH)
display(scan[rank_cols].head(20))
